# CS5224 Data Curation — Stage 2C (Automated / Semi-Automated, Revised QA Logic)

## Automated identity decisions, independent QA, fulfilment review, and provisional canonical products

This revision keeps the automated Stage 2C identity-matching workflow, but corrects the QA interpretation.

### Key QA correction

For CartOptix product identity, these decisions belong to the same **NON_MATCH** family:

- `VARIANT_MISMATCH`
- `NOT_SAME_PRODUCT`
- `EXCLUDE_OTHER`

Therefore:

```text
AUTO_NOT_SAME_PRODUCT
vs
QA VARIANT_MISMATCH
```

is **not** a material identity conflict. Both decisions mean that the two retailer listings must not be merged into one canonical product.

A **material QA conflict** occurs only when the decision family changes, for example:

```text
AUTO NOT-MATCH
vs
QA MATCH_CONFIRMED
```

or:

```text
AUTO MATCH_CONFIRMED
vs
QA NOT-MATCH
```

### Decision families

- `MATCH`
- `NON_MATCH`
- `UNRESOLVED`

### Auditability

A reviewed QA row is considered complete only when it contains:

- `qa_identity_decision`
- `qa_reviewer`
- `qa_evidence`

QA label differences within the same family are recorded separately from material conflicts.

Material conflicts are written to a dedicated conflict-resolution working file and are excluded from canonical grouping until resolved.

### Other Stage 2C principles

- Automated rules favor precision over recall.
- Borderline candidate pairs remain in the manual-review queue.
- Product identity remains independent of stock status.
- Direct supermarket fulfilment remains a separate listing-level verification task.
- Existing manual/QA decisions are preserved across notebook reruns.


## 1. Imports and paths

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import hashlib
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CURATED_DATA_DIR = PROJECT_ROOT / "data" / "curated"

STRICT_CANDIDATES_FILE = (
    CURATED_DATA_DIR
    / "stage2b_strict_candidate_pairs.csv"
)

PACK_DISCREPANCY_FILE = (
    CURATED_DATA_DIR
    / "stage2b_pack_discrepancy_review.csv"
)

ENRICHED_PILOT_FILE = (
    CURATED_DATA_DIR
    / "stage2b_pilot_enriched.csv"
)

for file_path in [
    STRICT_CANDIDATES_FILE,
    PACK_DISCREPANCY_FILE,
    ENRICHED_PILOT_FILE,
]:
    print(
        f"{file_path.name:48s} -> "
        f"{file_path.exists()}"
    )
    assert file_path.exists()

print(
    "\nPASS: Required Stage 2B inputs found."
)


stage2b_strict_candidate_pairs.csv               -> True
stage2b_pack_discrepancy_review.csv              -> True
stage2b_pilot_enriched.csv                       -> True

PASS: Required Stage 2B inputs found.


## 2. Load Stage 2B outputs

In [2]:
strict_candidates = pd.read_csv(
    STRICT_CANDIDATES_FILE,
    low_memory=False,
)

pack_discrepancy = pd.read_csv(
    PACK_DISCREPANCY_FILE,
    low_memory=False,
)

pilot = pd.read_csv(
    ENRICHED_PILOT_FILE,
    low_memory=False,
)

print(
    "Strict candidate pairs:",
    f"{len(strict_candidates):,}"
)

print(
    "Pack-discrepancy pairs:",
    f"{len(pack_discrepancy):,}"
)

print(
    "Pilot catalog records:",
    f"{len(pilot):,}"
)

display(
    strict_candidates.head(3)
)


Strict candidate pairs: 10,650
Pack-discrepancy pairs: 281
Pilot catalog records: 31,595


,candidate_pair_id,retailer_pair,target_category,brand_normalized,strict_pack_key,left_retailer,left_source_record_id,left_product_name,left_name_match_text,left_specification,...,right_offer_eligible,right_product_url,token_jaccard,sequence_similarity,combined_similarity,exact_name_core,candidate_priority,review_recommendation,final_match_verified,verification_notes
0,redmart_00000009__fairprice_00037226,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,True,https://www.fairprice.com.sg/product/arla-prot...,0.600000,0.539683,0.575873,False,LOW,LOW_SIMILARITY_REVIEW,NaN,NaN
1,redmart_00000009__fairprice_00037233,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.588235,0.406723,False,LOW,LOW_SIMILARITY_REVIEW,NaN,NaN
2,redmart_00000009__fairprice_00037232,redmart__fairprice,Snacks,arla,g|1|200.000000|200.000000,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,protein chocolate pudding lactose free,200 g,...,True,https://www.fairprice.com.sg/product/arla-prot...,0.285714,0.558824,0.394958,False,LOW,LOW_SIMILARITY_REVIEW,NaN,NaN


## 3. Validate Stage 2B inputs

In [3]:
REQUIRED_PAIR_COLUMNS = [
    "candidate_pair_id",
    "retailer_pair",
    "target_category",
    "brand_normalized",
    "strict_pack_key",

    "left_retailer",
    "left_source_record_id",
    "left_product_name",
    "left_name_match_text",
    "left_specification",
    "left_stock_status",
    "left_offer_eligible",
    "left_product_url",

    "right_retailer",
    "right_source_record_id",
    "right_product_name",
    "right_name_match_text",
    "right_specification",
    "right_stock_status",
    "right_offer_eligible",
    "right_product_url",

    "token_jaccard",
    "sequence_similarity",
    "combined_similarity",
    "exact_name_core",
    "candidate_priority",
    "review_recommendation",
]

missing_pair_columns = [
    column
    for column in REQUIRED_PAIR_COLUMNS
    if column not in strict_candidates.columns
]

assert not missing_pair_columns, (
    f"Missing Stage 2B columns: {missing_pair_columns}"
)

assert (
    strict_candidates[
        "candidate_pair_id"
    ]
    .isna()
    .sum()
    == 0
)

assert (
    strict_candidates[
        "candidate_pair_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    strict_candidates[
        "left_source_record_id"
    ]
    != strict_candidates[
        "right_source_record_id"
    ]
).all()

assert (
    strict_candidates[
        "left_retailer"
    ]
    != strict_candidates[
        "right_retailer"
    ]
).all()

print(
    "PASS: Stage 2B candidate inputs validated."
)


PASS: Stage 2B candidate inputs validated.


## 4. Configure conservative automation thresholds

These thresholds intentionally favor precision over recall.

A pair not meeting a high-confidence automatic rule is sent to manual review rather than guessed.


In [4]:
AUTO_CONFIRM_JACCARD = 0.92
AUTO_CONFIRM_SEQUENCE = 0.90
AUTO_CONFIRM_COMBINED = 0.92

AUTO_REJECT_JACCARD = 0.20
AUTO_REJECT_COMBINED = 0.35

MIN_TOKENS_FOR_LOW_SIM_REJECT = 2

INDEPENDENT_QA_SAMPLE_PER_AUTO_CLASS = 30


## 5. Define variant dimensions

A mismatch is automatically declared only when **both names explicitly contain conflicting values from the same variant dimension**.

If only one side contains a variant term, the pair is not auto-rejected.


In [5]:
VARIANT_DIMENSIONS = {
    "sugar_style": {
        "ZERO_SUGAR": [
            "zero sugar",
            "sugar free",
            "sugar-free",
            "no sugar",
            "no-sugar",
        ],
        "LESS_SUGAR": [
            "less sugar",
            "reduced sugar",
            "lower sugar",
        ],
        "ORIGINAL_REGULAR": [
            "original",
            "regular",
            "classic",
        ],
    },

    "diet_style": {
        "DIET": [
            "diet",
        ],
        "LIGHT": [
            "light",
            "lite",
        ],
        "REGULAR": [
            "regular",
            "original",
            "classic",
        ],
    },

    "fruit_flavour": {
        "LEMON": ["lemon"],
        "LIME": ["lime"],
        "ORANGE": ["orange"],
        "APPLE": ["apple"],
        "GRAPE": ["grape"],
        "PEACH": ["peach"],
        "LYCHEE": ["lychee"],
        "MANGO": ["mango"],
        "STRAWBERRY": ["strawberry"],
        "BLUEBERRY": ["blueberry"],
        "RASPBERRY": ["raspberry"],
        "BERRY": ["berry"],
        "PINEAPPLE": ["pineapple"],
        "WATERMELON": ["watermelon"],
    },

    "sweet_flavour": {
        "CHOCOLATE": [
            "chocolate",
            "choco",
        ],
        "VANILLA": ["vanilla"],
        "CARAMEL": ["caramel"],
        "MATCHA": ["matcha"],
        "MOCHA": ["mocha"],
        "COFFEE": ["coffee"],
    },

    "snack_seasoning": {
        "ORIGINAL": [
            "original",
            "classic",
            "plain",
        ],
        "BBQ": [
            "bbq",
            "barbecue",
        ],
        "CHEESE": ["cheese"],
        "SEAWEED": ["seaweed"],
        "SOUR_CREAM": [
            "sour cream",
        ],
        "SALTED": ["salted"],
        "UNSALTED": ["unsalted"],
        "SPICY": [
            "spicy",
            "hot",
            "chilli",
            "chili",
        ],
        "TRUFFLE": ["truffle"],
        "GARLIC": ["garlic"],
        "HONEY": ["honey"],
    },

    "protein_flavour": {
        "CHICKEN": ["chicken"],
        "BEEF": ["beef"],
        "PORK": ["pork"],
        "TUNA": ["tuna"],
        "SALMON": ["salmon"],
        "SARDINE": ["sardine"],
        "MACKEREL": ["mackerel"],
    },

    "tea_style": {
        "GREEN_TEA": [
            "green tea",
        ],
        "BLACK_TEA": [
            "black tea",
        ],
        "MILK_TEA": [
            "milk tea",
        ],
        "OOLONG": ["oolong"],
        "EARL_GREY": [
            "earl grey",
        ],
    },
}


## 6. Define helper functions for automated identity assessment

In [6]:
GENERIC_EXTRA_TOKENS = {
    "drink",
    "drinks",
    "beverage",
    "beverages",
    "snack",
    "snacks",
    "food",
    "foods",
    "can",
    "cans",
    "bottle",
    "bottles",
    "packet",
    "pack",
    "packs",
    "bag",
    "bags",
    "box",
    "boxes",
    "premium",
    "new",
    "imported",
    "value",
    "family",
    "size",
    "originally",
}


def clean_identity_text(text):
    if pd.isna(text):
        return ""

    text = str(text).casefold()
    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def token_set(text):
    return set(
        clean_identity_text(
            text
        ).split()
    )


def detect_variant_values(text):
    normalized = clean_identity_text(
        text
    )

    detected = {}

    for (
        dimension,
        values
    ) in VARIANT_DIMENSIONS.items():
        found_values = set()

        for (
            value_name,
            phrases
        ) in values.items():
            for phrase in phrases:
                phrase_normalized = (
                    clean_identity_text(
                        phrase
                    )
                )

                pattern = (
                    r"(?<![a-z0-9])"
                    + re.escape(
                        phrase_normalized
                    )
                    + r"(?![a-z0-9])"
                )

                if re.search(
                    pattern,
                    normalized
                ):
                    found_values.add(
                        value_name
                    )
                    break

        if found_values:
            detected[
                dimension
            ] = found_values

    return detected


def find_variant_conflicts(
    left_text,
    right_text,
):
    left_variants = (
        detect_variant_values(
            left_text
        )
    )

    right_variants = (
        detect_variant_values(
            right_text
        )
    )

    conflicts = []

    common_dimensions = (
        set(left_variants)
        &
        set(right_variants)
    )

    for dimension in sorted(
        common_dimensions
    ):
        left_values = (
            left_variants[
                dimension
            ]
        )

        right_values = (
            right_variants[
                dimension
            ]
        )

        if left_values.isdisjoint(
            right_values
        ):
            conflicts.append(
                {
                    "dimension":
                        dimension,
                    "left_values":
                        sorted(
                            left_values
                        ),
                    "right_values":
                        sorted(
                            right_values
                        ),
                }
            )

    return conflicts


def generic_containment_match(
    left_text,
    right_text,
):
    left_tokens = token_set(
        left_text
    )

    right_tokens = token_set(
        right_text
    )

    if (
        len(left_tokens) < 2
        or len(right_tokens) < 2
    ):
        return False

    if left_tokens <= right_tokens:
        extras = (
            right_tokens
            - left_tokens
        )

    elif right_tokens <= left_tokens:
        extras = (
            left_tokens
            - right_tokens
        )

    else:
        return False

    return (
        len(extras) <= 2
        and extras <= GENERIC_EXTRA_TOKENS
    )


def format_variant_conflicts(
    conflicts
):
    parts = []

    for conflict in conflicts:
        parts.append(
            (
                f"{conflict['dimension']}: "
                f"{'/'.join(conflict['left_values'])} "
                f"vs "
                f"{'/'.join(conflict['right_values'])}"
            )
        )

    return "; ".join(parts)


## 7. Automated identity decision function

Rule order matters:

1. explicit variant conflict;
2. exact normalized name core;
3. identical token set;
4. very high lexical similarity;
5. conservative containment with generic extra wording;
6. very low similarity rejection;
7. otherwise manual review.


In [7]:
def automated_identity_decision(row):
    left_text = clean_identity_text(
        row[
            "left_name_match_text"
        ]
    )

    right_text = clean_identity_text(
        row[
            "right_name_match_text"
        ]
    )

    conflicts = find_variant_conflicts(
        row["left_product_name"],
        row["right_product_name"],
    )

    if conflicts:
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_VARIANT_MISMATCH",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R1_EXPLICIT_VARIANT_CONFLICT",
                "auto_decision_reason":
                    (
                        "Both listings contain explicit "
                        "conflicting variant terms: "
                        + format_variant_conflicts(
                            conflicts
                        )
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        left_text
        and right_text
        and left_text == right_text
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R2_EXACT_NORMALIZED_CORE",
                "auto_decision_reason":
                    (
                        "Normalized product identity text "
                        "is exactly equal after brand and "
                        "pack wording removal."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    left_tokens = token_set(
        left_text
    )

    right_tokens = token_set(
        right_text
    )

    if (
        left_tokens
        and right_tokens
        and left_tokens
        == right_tokens
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R3_IDENTICAL_TOKEN_SET",
                "auto_decision_reason":
                    (
                        "Product identity token sets are "
                        "identical; ordering/wording differs."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    jaccard = float(
        row[
            "token_jaccard"
        ]
    )

    sequence = float(
        row[
            "sequence_similarity"
        ]
    )

    combined = float(
        row[
            "combined_similarity"
        ]
    )

    if (
        jaccard >= AUTO_CONFIRM_JACCARD
        and sequence >= AUTO_CONFIRM_SEQUENCE
        and combined >= AUTO_CONFIRM_COMBINED
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R4_VERY_HIGH_NAME_SIMILARITY",
                "auto_decision_reason":
                    (
                        "Same brand and strict pack from "
                        "Stage 2B, no detected variant "
                        "conflict, and very high lexical "
                        "agreement."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        generic_containment_match(
            left_text,
            right_text,
        )
        and sequence >= 0.80
        and combined >= 0.80
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_MATCH_CONFIRMED",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R5_GENERIC_WORDING_CONTAINMENT",
                "auto_decision_reason":
                    (
                        "One identity name is contained in "
                        "the other and the only extra tokens "
                        "are generic listing descriptors."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    if (
        len(left_tokens)
        >= MIN_TOKENS_FOR_LOW_SIM_REJECT
        and len(right_tokens)
        >= MIN_TOKENS_FOR_LOW_SIM_REJECT
        and jaccard <= AUTO_REJECT_JACCARD
        and combined <= AUTO_REJECT_COMBINED
    ):
        return pd.Series(
            {
                "auto_identity_decision":
                    "AUTO_NOT_SAME_PRODUCT",
                "auto_confidence":
                    "HIGH",
                "auto_rule_id":
                    "R6_VERY_LOW_NAME_SIMILARITY",
                "auto_decision_reason":
                    (
                        "Same brand and pack block, but "
                        "product identity names have very "
                        "low token and combined similarity."
                    ),
                "requires_manual_review":
                    False,
            }
        )

    return pd.Series(
        {
            "auto_identity_decision":
                "NEEDS_MANUAL_REVIEW",
            "auto_confidence":
                "REVIEW",
            "auto_rule_id":
                "R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE",
            "auto_decision_reason":
                (
                    "Automated evidence is not strong "
                    "enough for a high-confidence decision."
                ),
            "requires_manual_review":
                True,
        }
    )


## 8. Run automated identity assessment

In [8]:
automation_result = (
    strict_candidates
    .apply(
        automated_identity_decision,
        axis=1,
    )
)

automated_pairs = pd.concat(
    [
        strict_candidates.copy(),
        automation_result,
    ],
    axis=1,
)

print(
    "Automated identity decision counts:"
)

print(
    automated_pairs[
        "auto_identity_decision"
    ]
    .value_counts()
)

print(
    "\nManual review required:"
)

print(
    automated_pairs[
        "requires_manual_review"
    ]
    .value_counts()
)

display(
    automated_pairs[
        [
            "candidate_pair_id",
            "left_product_name",
            "right_product_name",
            "token_jaccard",
            "sequence_similarity",
            "combined_similarity",
            "auto_identity_decision",
            "auto_confidence",
            "auto_rule_id",
            "auto_decision_reason",
        ]
    ].head(20)
)


Automated identity decision counts:
auto_identity_decision
NEEDS_MANUAL_REVIEW      7026
AUTO_NOT_SAME_PRODUCT    1602
AUTO_VARIANT_MISMATCH    1112
AUTO_MATCH_CONFIRMED      910
Name: count, dtype: int64

Manual review required:
requires_manual_review
True     7026
False    3624
Name: count, dtype: int64


,candidate_pair_id,left_product_name,right_product_name,token_jaccard,sequence_similarity,combined_similarity,auto_identity_decision,auto_confidence,auto_rule_id,auto_decision_reason
0,redmart_00000009__fairprice_00037226,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Chocolate,0.600000,0.539683,0.575873,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
1,redmart_00000009__fairprice_00037233,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Hazelnut Latte,0.285714,0.588235,0.406723,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
2,redmart_00000009__fairprice_00037232,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla Protein Pudding - Salted Caramel,0.285714,0.558824,0.394958,AUTO_VARIANT_MISMATCH,HIGH,R1_EXPLICIT_VARIANT_CONFLICT,Both listings contain explicit conflicting var...
3,redmart_00000011__fairprice_00037233,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Hazelnut Latte,0.500000,0.626866,0.550746,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
4,redmart_00000011__fairprice_00037226,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Chocolate,0.333333,0.645161,0.458065,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
5,redmart_00000011__fairprice_00037232,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla Protein Pudding - Salted Caramel,0.285714,0.567164,0.398294,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
6,redmart_00000036__fairprice_00037232,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Salted Caramel,0.666667,0.602740,0.641096,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
7,redmart_00000036__fairprice_00037226,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Chocolate,0.285714,0.588235,0.406723,AUTO_VARIANT_MISMATCH,HIGH,R1_EXPLICIT_VARIANT_CONFLICT,Both listings contain explicit conflicting var...
8,redmart_00000036__fairprice_00037233,Arla Protein Salted Caramel Pudding 200G - Lac...,Arla Protein Pudding - Hazelnut Latte,0.250000,0.547945,0.369178,NEEDS_MANUAL_REVIEW,REVIEW,R7_BORDERLINE_OR_INSUFFICIENT_EVIDENCE,Automated evidence is not strong enough for a ...
9,redmart_00000085__fairprice_00037226,Arla Protein Vanilla Cookie Pudding 200G - Lac...,Arla Protein Pudding - Chocolate,0.285714,0.588235,0.406723,AUTO_VARIANT_MISMATCH,HIGH,R1_EXPLICIT_VARIANT_CONFLICT,Both listings contain explicit conflicting var...


## 9. Validate automated decision invariants

In [9]:
VALID_AUTO_DECISIONS = {
    "AUTO_MATCH_CONFIRMED",
    "AUTO_VARIANT_MISMATCH",
    "AUTO_NOT_SAME_PRODUCT",
    "NEEDS_MANUAL_REVIEW",
}

actual_auto_decisions = set(
    automated_pairs[
        "auto_identity_decision"
    ]
    .dropna()
    .unique()
)

assert (
    actual_auto_decisions
    <= VALID_AUTO_DECISIONS
)

assert (
    automated_pairs.loc[
        automated_pairs[
            "auto_identity_decision"
        ]
        == "NEEDS_MANUAL_REVIEW",
        "requires_manual_review"
    ]
    .eq(True)
    .all()
)

assert (
    automated_pairs.loc[
        automated_pairs[
            "auto_identity_decision"
        ]
        != "NEEDS_MANUAL_REVIEW",
        "requires_manual_review"
    ]
    .eq(False)
    .all()
)

print(
    "PASS: Automated decision invariants succeeded."
)


PASS: Automated decision invariants succeeded.


## 10. Create or refresh manual-review working file

Only borderline pairs enter this file.

Existing human decisions are preserved across reruns.


In [10]:
MANUAL_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2c_manual_identity_review_working.csv"
)

manual_review_base = (
    automated_pairs.loc[
        automated_pairs[
            "requires_manual_review"
        ]
    ]
    .copy()
)

MANUAL_DECISION_COLUMNS = [
    "manual_identity_decision",
    "manual_reason",
    "manual_reviewer",
    "manual_evidence",
    "manual_notes",
]

manual_review_base[
    "manual_identity_decision"
] = "UNREVIEWED"

for column in MANUAL_DECISION_COLUMNS[1:]:
    manual_review_base[column] = pd.NA

if MANUAL_REVIEW_FILE.exists():
    existing_manual_review = pd.read_csv(
        MANUAL_REVIEW_FILE,
        dtype="string",
        low_memory=False,
    )

    existing_manual_values = (
        existing_manual_review[
            [
                "candidate_pair_id",
                *MANUAL_DECISION_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
    )

    manual_review = (
        manual_review_base
        .drop(
            columns=MANUAL_DECISION_COLUMNS
        )
        .merge(
            existing_manual_values,
            on="candidate_pair_id",
            how="left",
        )
    )

    manual_review[
        "manual_identity_decision"
    ] = (
        manual_review[
            "manual_identity_decision"
        ]
        .fillna("UNREVIEWED")
    )

else:
    manual_review = (
        manual_review_base.copy()
    )

manual_review.to_csv(
    MANUAL_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Manual identity review pairs:",
    f"{len(manual_review):,}"
)

print(
    "\nManual decision counts:"
)

print(
    manual_review[
        "manual_identity_decision"
    ]
    .value_counts(
        dropna=False
    )
)


Manual identity review pairs: 7,026

Manual decision counts:
manual_identity_decision
UNREVIEWED    7026
Name: count, dtype: Int64


## 11. Create deterministic independent-QA sample

A fixed sample is drawn from each automated decision class.

This supports the project requirement that samples of automated decisions are independently reviewed.

The sample is stable across reruns because the ordering is based on a SHA-256 hash of `candidate_pair_id`.


In [11]:
def stable_hash_value(value):
    digest = hashlib.sha256(
        str(value).encode(
            "utf-8"
        )
    ).hexdigest()

    return int(
        digest[:16],
        16
    )


auto_only = automated_pairs.loc[
    ~automated_pairs[
        "requires_manual_review"
    ]
].copy()

auto_only[
    "_qa_hash"
] = (
    auto_only[
        "candidate_pair_id"
    ]
    .map(
        stable_hash_value
    )
)

qa_sample_parts = []

for (
    decision,
    group
) in auto_only.groupby(
    "auto_identity_decision"
):
    sample_n = min(
        INDEPENDENT_QA_SAMPLE_PER_AUTO_CLASS,
        len(group),
    )

    qa_sample_parts.append(
        group
        .sort_values(
            "_qa_hash"
        )
        .head(
            sample_n
        )
    )

if qa_sample_parts:
    qa_sample_base = pd.concat(
        qa_sample_parts,
        ignore_index=True,
    )
else:
    qa_sample_base = (
        auto_only.head(0).copy()
    )

qa_sample_base = qa_sample_base.drop(
    columns=[
        "_qa_hash"
    ],
    errors="ignore",
)

QA_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2c_independent_qa_review_working.csv"
)

QA_MANUAL_COLUMNS = [
    "qa_identity_decision",
    "qa_reviewer",
    "qa_evidence",
    "qa_notes",
]

qa_sample_base[
    "qa_identity_decision"
] = "UNREVIEWED"

for column in QA_MANUAL_COLUMNS[1:]:
    qa_sample_base[column] = pd.NA

if QA_REVIEW_FILE.exists():
    existing_qa = pd.read_csv(
        QA_REVIEW_FILE,
        dtype="string",
        low_memory=False,
    )

    existing_qa_values = (
        existing_qa[
            [
                "candidate_pair_id",
                *QA_MANUAL_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
    )

    qa_review = (
        qa_sample_base
        .drop(
            columns=QA_MANUAL_COLUMNS
        )
        .merge(
            existing_qa_values,
            on="candidate_pair_id",
            how="left",
        )
    )

    qa_review[
        "qa_identity_decision"
    ] = (
        qa_review[
            "qa_identity_decision"
        ]
        .fillna("UNREVIEWED")
    )

else:
    qa_review = (
        qa_sample_base.copy()
    )

qa_review.to_csv(
    QA_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Independent QA sample rows:",
    f"{len(qa_review):,}"
)

print(
    "\nQA sample by automated class:"
)

print(
    qa_review[
        "auto_identity_decision"
    ]
    .value_counts()
)


Independent QA sample rows: 90

QA sample by automated class:
auto_identity_decision
AUTO_MATCH_CONFIRMED     30
AUTO_NOT_SAME_PRODUCT    30
AUTO_VARIANT_MISMATCH    30
Name: count, dtype: int64


## 12. Define allowed human decision vocabulary

In [12]:
VALID_HUMAN_DECISIONS = {
    "UNREVIEWED",
    "MATCH_CONFIRMED",
    "VARIANT_MISMATCH",
    "NOT_SAME_PRODUCT",
    "NEEDS_MORE_EVIDENCE",
    "EXCLUDE_OTHER",
}

manual_invalid = sorted(
    set(
        manual_review[
            "manual_identity_decision"
        ]
        .dropna()
        .unique()
    )
    - VALID_HUMAN_DECISIONS
)

qa_invalid = sorted(
    set(
        qa_review[
            "qa_identity_decision"
        ]
        .dropna()
        .unique()
    )
    - VALID_HUMAN_DECISIONS
)

print(
    "Invalid manual decisions:",
    manual_invalid
)

print(
    "Invalid QA decisions:",
    qa_invalid
)

assert not manual_invalid
assert not qa_invalid

print(
    "\nPASS: Human review vocabularies validated."
)


Invalid manual decisions: []
Invalid QA decisions: []

PASS: Human review vocabularies validated.


## 13. Resolve automated and human decisions using decision families

The resolution order is:

1. Human decision for a pair already sent to the manual-review queue;
2. Complete independent QA review;
3. Automated high-confidence decision.

QA agreement is evaluated at the **decision-family level**, not merely by exact label.

Examples:

```text
AUTO_NOT_SAME_PRODUCT + QA VARIANT_MISMATCH
→ same NON_MATCH family
→ QA agreement
```

```text
AUTO_NOT_SAME_PRODUCT + QA MATCH_CONFIRMED
→ NON_MATCH vs MATCH
→ material QA conflict
```

A QA row is used only when the reviewer and evidence fields are also completed.


In [13]:
AUTO_TO_FINAL = {
    "AUTO_MATCH_CONFIRMED":
        "MATCH_CONFIRMED",
    "AUTO_VARIANT_MISMATCH":
        "VARIANT_MISMATCH",
    "AUTO_NOT_SAME_PRODUCT":
        "NOT_SAME_PRODUCT",
    "NEEDS_MANUAL_REVIEW":
        "UNREVIEWED",
}

DECISION_FAMILY_MAP = {
    "MATCH_CONFIRMED":
        "MATCH",

    "VARIANT_MISMATCH":
        "NON_MATCH",
    "NOT_SAME_PRODUCT":
        "NON_MATCH",
    "EXCLUDE_OTHER":
        "NON_MATCH",

    "NEEDS_MORE_EVIDENCE":
        "UNRESOLVED",
    "UNREVIEWED":
        "UNRESOLVED",
}


def decision_family(decision):
    if pd.isna(decision):
        return "UNRESOLVED"

    return DECISION_FAMILY_MAP.get(
        str(decision),
        "UNRESOLVED",
    )


def has_nonblank_text(series):
    return (
        series
        .notna()
        &
        series
        .astype("string")
        .str.strip()
        .ne("")
    )


manual_lookup = (
    manual_review[
        [
            "candidate_pair_id",
            "manual_identity_decision",
            "manual_reason",
            "manual_reviewer",
            "manual_evidence",
            "manual_notes",
        ]
    ]
    .drop_duplicates(
        subset=[
            "candidate_pair_id"
        ]
    )
)

qa_lookup = (
    qa_review[
        [
            "candidate_pair_id",
            "qa_identity_decision",
            "qa_reviewer",
            "qa_evidence",
            "qa_notes",
        ]
    ]
    .drop_duplicates(
        subset=[
            "candidate_pair_id"
        ]
    )
)

final_pairs = (
    automated_pairs
    .merge(
        manual_lookup,
        on="candidate_pair_id",
        how="left",
    )
    .merge(
        qa_lookup,
        on="candidate_pair_id",
        how="left",
    )
)

# ---------------------------------------------------------
# Automated decision converted to the common final vocabulary.
# ---------------------------------------------------------

final_pairs[
    "auto_final_decision"
] = (
    final_pairs[
        "auto_identity_decision"
    ]
    .map(
        AUTO_TO_FINAL
    )
    .astype("string")
)

final_pairs[
    "auto_decision_family"
] = (
    final_pairs[
        "auto_final_decision"
    ]
    .map(
        decision_family
    )
    .astype("string")
)

# ---------------------------------------------------------
# Evaluate independent-QA completeness.
# ---------------------------------------------------------

final_pairs[
    "qa_has_decision"
] = (
    final_pairs[
        "qa_identity_decision"
    ]
    .notna()
    &
    (
        final_pairs[
            "qa_identity_decision"
        ]
        != "UNREVIEWED"
    )
)

final_pairs[
    "qa_reviewer_present"
] = has_nonblank_text(
    final_pairs[
        "qa_reviewer"
    ]
)

final_pairs[
    "qa_evidence_present"
] = has_nonblank_text(
    final_pairs[
        "qa_evidence"
    ]
)

final_pairs[
    "qa_review_complete"
] = (
    final_pairs[
        "qa_has_decision"
    ]
    &
    final_pairs[
        "qa_reviewer_present"
    ]
    &
    final_pairs[
        "qa_evidence_present"
    ]
)

final_pairs[
    "qa_review_incomplete"
] = (
    final_pairs[
        "qa_has_decision"
    ]
    &
    ~final_pairs[
        "qa_review_complete"
    ]
)

final_pairs[
    "qa_decision_family"
] = (
    final_pairs[
        "qa_identity_decision"
    ]
    .map(
        decision_family
    )
    .astype("string")
)

# ---------------------------------------------------------
# Distinguish exact-label agreement from family agreement.
# ---------------------------------------------------------

final_pairs[
    "qa_exact_label_agreement"
] = (
    final_pairs[
        "qa_review_complete"
    ]
    &
    (
        final_pairs[
            "qa_identity_decision"
        ]
        == final_pairs[
            "auto_final_decision"
        ]
    )
)

final_pairs[
    "qa_family_agreement"
] = (
    final_pairs[
        "qa_review_complete"
    ]
    &
    (
        final_pairs[
            "qa_decision_family"
        ]
        == final_pairs[
            "auto_decision_family"
        ]
    )
)

final_pairs[
    "qa_same_family_label_difference"
] = (
    final_pairs[
        "qa_family_agreement"
    ]
    &
    ~final_pairs[
        "qa_exact_label_agreement"
    ]
)

final_pairs[
    "qa_material_conflict"
] = (
    final_pairs[
        "qa_review_complete"
    ]
    &
    (
        final_pairs[
            "qa_decision_family"
        ]
        != final_pairs[
            "auto_decision_family"
        ]
    )
)

# ---------------------------------------------------------
# Start final resolution from the automated decision.
# ---------------------------------------------------------

final_pairs[
    "final_identity_decision"
] = (
    final_pairs[
        "auto_final_decision"
    ]
    .astype("string")
)

final_pairs[
    "final_decision_source"
] = "AUTOMATED"

# ---------------------------------------------------------
# Manual review of borderline pairs overrides automation.
# ---------------------------------------------------------

manual_override_mask = (
    final_pairs[
        "manual_identity_decision"
    ]
    .notna()
    &
    (
        final_pairs[
            "manual_identity_decision"
        ]
        != "UNREVIEWED"
    )
)

final_pairs.loc[
    manual_override_mask,
    "final_identity_decision"
] = final_pairs.loc[
    manual_override_mask,
    "manual_identity_decision"
]

final_pairs.loc[
    manual_override_mask,
    "final_decision_source"
] = "MANUAL_REVIEW"

# ---------------------------------------------------------
# Apply complete QA reviews only when no manual override exists.
#
# Exact label agreement:
#   keep the automated decision and mark it QA-confirmed.
#
# Same family, different label:
#   adopt the QA label because it is a more specific human-reviewed
#   explanation, but do NOT call it a conflict.
#
# Different family:
#   withhold the pair as REVIEW_CONFLICT until resolved.
# ---------------------------------------------------------

qa_exact_mask = (
    final_pairs[
        "qa_exact_label_agreement"
    ]
    &
    ~manual_override_mask
)

final_pairs.loc[
    qa_exact_mask,
    "final_decision_source"
] = "AUTOMATED_QA_CONFIRMED"

qa_same_family_mask = (
    final_pairs[
        "qa_same_family_label_difference"
    ]
    &
    ~manual_override_mask
)

final_pairs.loc[
    qa_same_family_mask,
    "final_identity_decision"
] = final_pairs.loc[
    qa_same_family_mask,
    "qa_identity_decision"
]

final_pairs.loc[
    qa_same_family_mask,
    "final_decision_source"
] = "QA_REVIEW_SAME_FAMILY"

qa_material_conflict_mask = (
    final_pairs[
        "qa_material_conflict"
    ]
    &
    ~manual_override_mask
)

final_pairs.loc[
    qa_material_conflict_mask,
    "final_identity_decision"
] = "REVIEW_CONFLICT"

final_pairs.loc[
    qa_material_conflict_mask,
    "final_decision_source"
] = "QA_MATERIAL_CONFLICT"

# ---------------------------------------------------------
# If a QA decision was entered but reviewer/evidence is missing,
# do not use the incomplete QA record as independent validation.
# The underlying automated decision remains, but the missing audit
# information is explicitly flagged for completion.
# ---------------------------------------------------------

print(
    "QA rows with a decision but incomplete "
    "reviewer/evidence fields:",
    f"{int(final_pairs['qa_review_incomplete'].sum()):,}"
)

print(
    "\nFinal identity decision counts "
    "before material-conflict resolution:"
)

print(
    final_pairs[
        "final_identity_decision"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\nFinal decision sources:"
)

print(
    final_pairs[
        "final_decision_source"
    ]
    .value_counts(
        dropna=False
    )
)


QA rows with a decision but incomplete reviewer/evidence fields: 0

Final identity decision counts before material-conflict resolution:
final_identity_decision
UNREVIEWED          7026
NOT_SAME_PRODUCT    1572
VARIANT_MISMATCH    1140
MATCH_CONFIRMED      910
REVIEW_CONFLICT        2
Name: count, dtype: Int64

Final decision sources:
final_decision_source
AUTOMATED                 10560
AUTOMATED_QA_CONFIRMED       60
QA_REVIEW_SAME_FAMILY        28
QA_MATERIAL_CONFLICT          2
Name: count, dtype: int64


## 14. Build QA family-agreement and material-conflict reports

The QA report separates:

- exact-label agreement;
- same-family agreement with a different rejection reason;
- material family conflicts;
- incomplete QA audit rows.

Only material family conflicts require conflict resolution.


In [14]:
qa_complete_rows = (
    final_pairs.loc[
        final_pairs[
            "qa_review_complete"
        ]
    ]
    .copy()
)

qa_exact_agreements = (
    qa_complete_rows.loc[
        qa_complete_rows[
            "qa_exact_label_agreement"
        ]
    ]
    .copy()
)

qa_same_family_differences = (
    qa_complete_rows.loc[
        qa_complete_rows[
            "qa_same_family_label_difference"
        ]
    ]
    .copy()
)

qa_material_conflicts = (
    qa_complete_rows.loc[
        qa_complete_rows[
            "qa_material_conflict"
        ]
    ]
    .copy()
)

qa_incomplete_reviews = (
    final_pairs.loc[
        final_pairs[
            "qa_review_incomplete"
        ]
    ]
    .copy()
)

# Backward-compatible alias used by some later cells.
qa_conflicts = (
    qa_material_conflicts.copy()
)

qa_complete_count = len(
    qa_complete_rows
)

qa_family_agreement_count = int(
    qa_complete_rows[
        "qa_family_agreement"
    ].sum()
)

qa_material_conflict_count = len(
    qa_material_conflicts
)

if qa_complete_count > 0:
    qa_family_agreement_rate = (
        qa_family_agreement_count
        / qa_complete_count
    )
else:
    qa_family_agreement_rate = np.nan

print(
    "Complete QA reviews:",
    f"{qa_complete_count:,}"
)

print(
    "Exact-label agreements:",
    f"{len(qa_exact_agreements):,}"
)

print(
    "Same-family label differences:",
    f"{len(qa_same_family_differences):,}"
)

print(
    "Material QA conflicts:",
    f"{qa_material_conflict_count:,}"
)

print(
    "Incomplete QA audit rows:",
    f"{len(qa_incomplete_reviews):,}"
)

if qa_complete_count > 0:
    print(
        "QA family-level agreement:",
        f"{qa_family_agreement_count:,}/"
        f"{qa_complete_count:,} "
        f"({qa_family_agreement_rate:.2%})"
    )

print(
    "\nSame-family label differences "
    "(not material conflicts):"
)

display(
    qa_same_family_differences[
        [
            "candidate_pair_id",
            "left_product_name",
            "right_product_name",
            "auto_final_decision",
            "auto_decision_family",
            "qa_identity_decision",
            "qa_decision_family",
            "qa_reviewer",
            "qa_evidence",
        ]
    ].head(30)
)

print(
    "\nMaterial QA conflicts requiring resolution:"
)

display(
    qa_material_conflicts[
        [
            "candidate_pair_id",
            "left_product_name",
            "right_product_name",
            "auto_final_decision",
            "auto_decision_family",
            "qa_identity_decision",
            "qa_decision_family",
            "qa_reviewer",
            "qa_evidence",
        ]
    ].head(30)
)


Complete QA reviews: 90
Exact-label agreements: 60
Same-family label differences: 28
Material QA conflicts: 2
Incomplete QA audit rows: 0
QA family-level agreement: 88/90 (97.78%)

Same-family label differences (not material conflicts):


,candidate_pair_id,left_product_name,right_product_name,auto_final_decision,auto_decision_family,qa_identity_decision,qa_decision_family,qa_reviewer,qa_evidence
723,redmart_00003221__coldstorage_00000709,Moccona Classic Medium Roast Intensity 5 Freez...,Moccona Espresso Instant Bold & Intense Coffee...,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1289,redmart_00004706__fairprice_00015225,Pokka Afternoon Straight Red Tea 1.5L,[BCRS] Pokka Bottle Drink - Honey Green Tea,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1314,redmart_00004757__fairprice_00015216,Pokka Peppermint Green Tea 1.5L,[BCRS] Pokka Bottle Drink - Jasmine Green Tea ...,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1397,redmart_00004745__coldstorage_00000876,Pokka Ice Blueberry Tea 1.5L,[BCRS] Pokka Oolong Tea 1.5L,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1503,fairprice_00015220__coldstorage_00001104,[BCRS] Pokka Premium Bottle Drink - Straight R...,[BCRS] Pokka Zero Sugar Peach Tea 1.5L,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1613,redmart_00005286__fairprice_00000799,Pokka Chrysanthemum White Tea (6 x 250ML),Pokka Packet Drink - Jasmine Green Tea,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1646,redmart_00004691__coldstorage_00000988,Pokka Afternoon Straight Red Tea (6 X 250ML),Pokka Oolong Tea 6 x 250ml,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1788,redmart_00004951__coldstorage_00001285,F&N Seasons Ice Peach Tea 1.5L,[BCRS] F&N Ice Cream Soda 1.5L,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
1885,fairprice_00000913__coldstorage_00000974,Yeo's Packet Drink - Lychee,Yeo's Winter Melon Drink 6 x 250ml,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant
2304,redmart_00005796__fairprice_00059293,Ceres Whispers of Summer 100% Juice 1L - No Su...,Ceres 100% Juice Blend Packet Drink - Orange,NOT_SAME_PRODUCT,NON_MATCH,VARIANT_MISMATCH,NON_MATCH,osh,variant



Material QA conflicts requiring resolution:


,candidate_pair_id,left_product_name,right_product_name,auto_final_decision,auto_decision_family,qa_identity_decision,qa_decision_family,qa_reviewer,qa_evidence
9560,redmart_00059472__coldstorage_00000523,Milo Original Chocolate Malt Powder With Milk ...,Milo Regular Malt Drink 900g,NOT_SAME_PRODUCT,NON_MATCH,MATCH_CONFIRMED,MATCH,osh,matched
10123,fairprice_00001135__coldstorage_00001113,Dilmah Pure Ceylon Tea Bags - English Breakfast,Dilmah Ceylon Teabags 100 x 2g,NOT_SAME_PRODUCT,NON_MATCH,MATCH_CONFIRMED,MATCH,osh,matched


## 15. Create or refresh material QA conflict-resolution working file

Only **material family conflicts** enter this file.

Examples:

```text
AUTO NON_MATCH vs QA MATCH
AUTO MATCH vs QA NON_MATCH
```

The resolution file preserves prior decisions across reruns.

Allowed resolution decisions use the same human identity vocabulary:

- `MATCH_CONFIRMED`
- `VARIANT_MISMATCH`
- `NOT_SAME_PRODUCT`
- `NEEDS_MORE_EVIDENCE`
- `EXCLUDE_OTHER`

A resolution is considered complete only when it includes a decision, reviewer and evidence.


In [15]:
QA_CONFLICT_RESOLUTION_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_conflict_resolution_working.csv"
)

QA_RESOLUTION_COLUMNS = [
    "resolution_identity_decision",
    "resolution_reviewer",
    "resolution_evidence",
    "resolution_notes",
]

qa_conflict_resolution_base = (
    qa_material_conflicts.copy()
)

qa_conflict_resolution_base[
    "resolution_identity_decision"
] = "UNREVIEWED"

for column in QA_RESOLUTION_COLUMNS[1:]:
    qa_conflict_resolution_base[
        column
    ] = pd.NA

if QA_CONFLICT_RESOLUTION_FILE.exists():
    existing_resolution = pd.read_csv(
        QA_CONFLICT_RESOLUTION_FILE,
        dtype="string",
        low_memory=False,
    )

    existing_resolution_values = (
        existing_resolution[
            [
                "candidate_pair_id",
                *QA_RESOLUTION_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
    )

    qa_conflict_resolution = (
        qa_conflict_resolution_base
        .drop(
            columns=QA_RESOLUTION_COLUMNS
        )
        .merge(
            existing_resolution_values,
            on="candidate_pair_id",
            how="left",
        )
    )

    qa_conflict_resolution[
        "resolution_identity_decision"
    ] = (
        qa_conflict_resolution[
            "resolution_identity_decision"
        ]
        .fillna("UNREVIEWED")
    )

else:
    qa_conflict_resolution = (
        qa_conflict_resolution_base.copy()
    )

qa_conflict_resolution.to_csv(
    QA_CONFLICT_RESOLUTION_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Material QA conflicts in resolution file:",
    f"{len(qa_conflict_resolution):,}"
)

print(
    "\nResolution decision counts:"
)

if len(qa_conflict_resolution) > 0:
    print(
        qa_conflict_resolution[
            "resolution_identity_decision"
        ]
        .value_counts(
            dropna=False
        )
    )
else:
    print(
        "No material conflicts."
    )


Material QA conflicts in resolution file: 2

Resolution decision counts:
resolution_identity_decision
MATCH_CONFIRMED    2
Name: count, dtype: Int64


## 16. Apply completed material-conflict resolutions

A material conflict remains excluded from canonical grouping until its resolution has:

- a non-`UNREVIEWED` decision;
- a reviewer;
- evidence.

Completed resolutions override the temporary `REVIEW_CONFLICT` state.


In [16]:
if len(qa_conflict_resolution) > 0:
    VALID_RESOLUTION_DECISIONS = (
        VALID_HUMAN_DECISIONS
    )

    invalid_resolution_decisions = sorted(
        set(
            qa_conflict_resolution[
                "resolution_identity_decision"
            ]
            .dropna()
            .unique()
        )
        - VALID_RESOLUTION_DECISIONS
    )

    assert not invalid_resolution_decisions, (
        "Invalid QA conflict-resolution decision(s): "
        f"{invalid_resolution_decisions}"
    )

    qa_conflict_resolution[
        "resolution_has_decision"
    ] = (
        qa_conflict_resolution[
            "resolution_identity_decision"
        ]
        .notna()
        &
        (
            qa_conflict_resolution[
                "resolution_identity_decision"
            ]
            != "UNREVIEWED"
        )
    )

    qa_conflict_resolution[
        "resolution_reviewer_present"
    ] = has_nonblank_text(
        qa_conflict_resolution[
            "resolution_reviewer"
        ]
    )

    qa_conflict_resolution[
        "resolution_evidence_present"
    ] = has_nonblank_text(
        qa_conflict_resolution[
            "resolution_evidence"
        ]
    )

    qa_conflict_resolution[
        "resolution_complete"
    ] = (
        qa_conflict_resolution[
            "resolution_has_decision"
        ]
        &
        qa_conflict_resolution[
            "resolution_reviewer_present"
        ]
        &
        qa_conflict_resolution[
            "resolution_evidence_present"
        ]
    )

    resolution_lookup = (
        qa_conflict_resolution[
            [
                "candidate_pair_id",
                "resolution_identity_decision",
                "resolution_reviewer",
                "resolution_evidence",
                "resolution_notes",
                "resolution_complete",
            ]
        ]
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
    )

    final_pairs = (
        final_pairs
        .merge(
            resolution_lookup,
            on="candidate_pair_id",
            how="left",
        )
    )

    completed_resolution_mask = (
        final_pairs[
            "resolution_complete"
        ]
        .fillna(False)
    )

    final_pairs.loc[
        completed_resolution_mask,
        "final_identity_decision"
    ] = final_pairs.loc[
        completed_resolution_mask,
        "resolution_identity_decision"
    ]

    final_pairs.loc[
        completed_resolution_mask,
        "final_decision_source"
    ] = "QA_CONFLICT_RESOLUTION"

else:
    qa_conflict_resolution[
        "resolution_complete"
    ] = pd.Series(
        dtype="boolean"
    )

    for column in [
        "resolution_identity_decision",
        "resolution_reviewer",
        "resolution_evidence",
        "resolution_notes",
        "resolution_complete",
    ]:
        final_pairs[column] = pd.NA


unresolved_material_conflicts = (
    final_pairs.loc[
        final_pairs[
            "final_identity_decision"
        ]
        == "REVIEW_CONFLICT"
    ]
    .copy()
)

print(
    "Completed material-conflict resolutions:",
    f"{int(final_pairs['final_decision_source'].eq('QA_CONFLICT_RESOLUTION').sum()):,}"
)

print(
    "Unresolved material QA conflicts:",
    f"{len(unresolved_material_conflicts):,}"
)


Completed material-conflict resolutions: 2
Unresolved material QA conflicts: 0


## 17. Extract final identity-confirmed pairs

These pairs can define canonical product identity.

Stock status is deliberately ignored here.


In [17]:
identity_confirmed_pairs = (
    final_pairs.loc[
        final_pairs[
            "final_identity_decision"
        ]
        == "MATCH_CONFIRMED"
    ]
    .copy()
)

print(
    "Final MATCH_CONFIRMED pairs:",
    f"{len(identity_confirmed_pairs):,}"
)

display(
    identity_confirmed_pairs[
        [
            "candidate_pair_id",
            "target_category",
            "brand_normalized",
            "left_retailer",
            "left_product_name",
            "left_specification",
            "right_retailer",
            "right_product_name",
            "right_specification",
            "final_decision_source",
        ]
    ].head(30)
)


Final MATCH_CONFIRMED pairs: 912


,candidate_pair_id,target_category,brand_normalized,left_retailer,left_product_name,left_specification,right_retailer,right_product_name,right_specification,final_decision_source
12,redmart_00000010__coldstorage_00004641,Snacks,fortune,redmart,Fortune Soya Beancurd Almond 360G,360 g,coldstorage,Fortune Almond Soya Beancurd 360g,360g,AUTOMATED
14,redmart_00000018__coldstorage_00004642,Snacks,fortune,redmart,Fortune Soya Beancurd Original 360G,360 g,coldstorage,Fortune Original Soya Beancurd 360g,360g,AUTOMATED
24,redmart_00000024__fairprice_00013004,Snacks,glico,redmart,Glico Pocky Choco Mint,65 g,fairprice,Glico Choco Mint Pocky,65 G,AUTOMATED
25,redmart_00000037__fairprice_00037252,Snacks,vatan se,redmart,Vatan Se Malai Rabri,200 g,fairprice,Vatan Se Malai Rabri,200 G,AUTOMATED
47,redmart_00000894__fairprice_00038596,Snacks,fairy port,redmart,Fairy Port Frozen Musang King Durian Mille Cre...,400 g,fairprice,Fairy Port Frozen Musang King Durian Mille Cre...,400 G,AUTOMATED
91,redmart_00001317__fairprice_00059772,Drinks,hi-c,redmart,HI-C Lemon Tea (Bundle Of 6),6 × 250 ml,fairprice,Hi-C Lemon Tea - Bundle of 6,6 x 250 ML,AUTOMATED
98,redmart_00001357__fairprice_00001157,Drinks,daribell,redmart,Daribell Pure Ginger Powder 180G,180 g,fairprice,Daribell Pure Ginger Powder,180g,AUTOMATED
101,redmart_00001414__fairprice_00060282,Drinks,redman,redmart,RedMan Assam Red Tea Leaf 600G,0.6 kg,fairprice,RedMan Assam Red Tea Leaf,600 G,AUTOMATED
107,redmart_00001769__fairprice_00059909,Drinks,redman,redmart,RedMan Jasmine Green Tea Leaf 600G,0.6 kg,fairprice,RedMan Jasmine Green Tea Leaf,600 G,AUTOMATED
110,redmart_00001487__fairprice_00059858,Drinks,ujinotsuyu,redmart,Ujinotsuyu Mugicha Barley Teapack,468 g,fairprice,Ujinotsuyu Mugicha Barley Teapack,468 G,AUTOMATED


## 18. Create or refresh fulfilment review for listings in confirmed identity pairs

Only listings that actually participate in confirmed identity pairs enter this working file.

Domain classification is provided as evidence assistance, but it does **not** automatically set `VERIFIED_DIRECT`.


In [18]:
confirmed_listing_ids = set(
    identity_confirmed_pairs[
        "left_source_record_id"
    ]
    .astype(str)
).union(
    set(
        identity_confirmed_pairs[
            "right_source_record_id"
        ]
        .astype(str)
    )
)

pilot_source_id = (
    pilot[
        "source_record_id"
    ]
    .astype(str)
)

fulfilment_base = (
    pilot.loc[
        pilot_source_id.isin(
            confirmed_listing_ids
        )
    ]
    .copy()
)

fulfilment_base[
    "source_record_id"
] = (
    fulfilment_base[
        "source_record_id"
    ]
    .astype(str)
)


def url_host(url):
    if pd.isna(url):
        return ""

    try:
        return (
            urlparse(
                str(url)
            )
            .netloc
            .casefold()
            .replace(
                "www.",
                "",
            )
        )
    except Exception:
        return ""


fulfilment_base[
    "source_url_host"
] = (
    fulfilment_base[
        "product_url_raw"
    ]
    .map(
        url_host
    )
)


def fulfilment_precheck(row):
    retailer = str(
        row[
            "retailer"
        ]
    ).casefold()

    host = str(
        row[
            "source_url_host"
        ]
    ).casefold()

    if (
        retailer == "fairprice"
        and (
            host == "fairprice.com.sg"
            or host.endswith(
                ".fairprice.com.sg"
            )
        )
    ):
        return (
            "OFFICIAL_RETAILER_DOMAIN_"
            "DIRECT_FULFILMENT_STILL_TO_VERIFY"
        )

    if (
        retailer == "coldstorage"
        and (
            host == "coldstorage.com.sg"
            or host.endswith(
                ".coldstorage.com.sg"
            )
        )
    ):
        return (
            "OFFICIAL_RETAILER_DOMAIN_"
            "DIRECT_FULFILMENT_STILL_TO_VERIFY"
        )

    if (
        retailer == "redmart"
        and (
            host == "lazada.sg"
            or host.endswith(
                ".lazada.sg"
            )
        )
    ):
        return (
            "MARKETPLACE_DOMAIN_"
            "SELLER_FULFILMENT_MUST_BE_VERIFIED"
        )

    return (
        "MANUAL_FULFILMENT_VERIFICATION_REQUIRED"
    )


fulfilment_base[
    "fulfilment_precheck"
] = (
    fulfilment_base
    .apply(
        fulfilment_precheck,
        axis=1,
    )
)

FULFILMENT_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2c_fulfilment_review_working.csv"
)

FULFILMENT_MANUAL_COLUMNS = [
    "fulfilment_status",
    "fulfilment_evidence",
    "fulfilment_reviewer",
    "fulfilment_notes",
]

fulfilment_base[
    "fulfilment_status"
] = "UNREVIEWED"

for column in (
    FULFILMENT_MANUAL_COLUMNS[1:]
):
    fulfilment_base[column] = pd.NA

if FULFILMENT_REVIEW_FILE.exists():
    existing_fulfilment = pd.read_csv(
        FULFILMENT_REVIEW_FILE,
        dtype="string",
        low_memory=False,
    )

    existing_values = (
        existing_fulfilment[
            [
                "source_record_id",
                *FULFILMENT_MANUAL_COLUMNS,
            ]
        ]
        .drop_duplicates(
            subset=[
                "source_record_id"
            ]
        )
    )

    fulfilment_review = (
        fulfilment_base
        .drop(
            columns=FULFILMENT_MANUAL_COLUMNS
        )
        .merge(
            existing_values,
            on="source_record_id",
            how="left",
        )
    )

    fulfilment_review[
        "fulfilment_status"
    ] = (
        fulfilment_review[
            "fulfilment_status"
        ]
        .fillna("UNREVIEWED")
    )

else:
    fulfilment_review = (
        fulfilment_base.copy()
    )

fulfilment_review.to_csv(
    FULFILMENT_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Fulfilment listings requiring verification:",
    f"{len(fulfilment_review):,}"
)

print(
    "\nFulfilment precheck:"
)

print(
    fulfilment_review[
        "fulfilment_precheck"
    ]
    .value_counts(
        dropna=False
    )
)


Fulfilment listings requiring verification: 1,720

Fulfilment precheck:
fulfilment_precheck
OFFICIAL_RETAILER_DOMAIN_DIRECT_FULFILMENT_STILL_TO_VERIFY    980
MARKETPLACE_DOMAIN_SELLER_FULFILMENT_MUST_BE_VERIFIED         740
Name: count, dtype: int64


## 19. Validate fulfilment vocabulary

In [19]:
VALID_FULFILMENT_STATUSES = {
    "UNREVIEWED",
    "VERIFIED_DIRECT",
    "NOT_DIRECT",
    "UNKNOWN",
}

invalid_fulfilment = sorted(
    set(
        fulfilment_review[
            "fulfilment_status"
        ]
        .dropna()
        .unique()
    )
    - VALID_FULFILMENT_STATUSES
)

print(
    "Invalid fulfilment statuses:",
    invalid_fulfilment
)

assert not invalid_fulfilment

print(
    "\nPASS: Fulfilment review vocabulary validated."
)


Invalid fulfilment statuses: []

PASS: Fulfilment review vocabulary validated.


## 20. Build provisional canonical identity groups

Canonical identities are based on product identity, not fulfilment or current stock.

This means an out-of-stock or not-yet-fulfilment-verified listing can still belong to the correct product identity group.


In [20]:
class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = {}

    def add(self, item):
        if item not in self.parent:
            self.parent[item] = item
            self.rank[item] = 0

    def find(self, item):
        self.add(item)

        if self.parent[item] != item:
            self.parent[item] = self.find(
                self.parent[item]
            )

        return self.parent[item]

    def union(self, left, right):
        root_left = self.find(left)
        root_right = self.find(right)

        if root_left == root_right:
            return

        left_rank = self.rank[
            root_left
        ]

        right_rank = self.rank[
            root_right
        ]

        if left_rank < right_rank:
            self.parent[
                root_left
            ] = root_right

        elif left_rank > right_rank:
            self.parent[
                root_right
            ] = root_left

        else:
            self.parent[
                root_right
            ] = root_left

            self.rank[
                root_left
            ] += 1


uf = UnionFind()

for row in (
    identity_confirmed_pairs
    .itertuples(
        index=False
    )
):
    uf.union(
        str(
            row.left_source_record_id
        ),
        str(
            row.right_source_record_id
        ),
    )


component_members = {}

for source_id in uf.parent:
    root = uf.find(
        source_id
    )

    component_members.setdefault(
        root,
        [],
    ).append(
        source_id
    )

print(
    "Provisional canonical components:",
    f"{len(component_members):,}"
)


Provisional canonical components: 833


## 21. Create canonical membership table

In [21]:
pilot_lookup = (
    pilot.copy()
)

pilot_lookup[
    "source_record_id"
] = (
    pilot_lookup[
        "source_record_id"
    ]
    .astype(str)
)

pilot_lookup = (
    pilot_lookup
    .set_index(
        "source_record_id",
        drop=False,
    )
)

membership_rows = []

for root, members in (
    component_members.items()
):
    members = sorted(
        set(
            str(member)
            for member in members
        )
    )

    stable_text = "||".join(
        members
    )

    canonical_hash = (
        hashlib.sha256(
            stable_text.encode(
                "utf-8"
            )
        )
        .hexdigest()[:12]
    )

    canonical_product_id = (
        f"CP_{canonical_hash}"
    )

    for source_id in members:
        if source_id not in (
            pilot_lookup.index
        ):
            continue

        row = pilot_lookup.loc[
            source_id
        ]

        membership_rows.append(
            {
                "canonical_product_id":
                    canonical_product_id,
                "source_record_id":
                    source_id,
                "retailer":
                    row[
                        "retailer"
                    ],
                "target_category":
                    row[
                        "target_category_final"
                    ],
                "brand_normalized":
                    row[
                        "brand_normalized"
                    ],
                "product_name_raw":
                    row[
                        "product_name_raw"
                    ],
                "specification_raw":
                    row[
                        "specification_raw"
                    ],
                "strict_pack_key":
                    row[
                        "strict_pack_key"
                    ],
                "product_price":
                    row[
                        "product_price"
                    ],
                "stock_status":
                    row[
                        "stock_status"
                    ],
                "currently_available":
                    row[
                        "currently_available"
                    ],
                "offer_eligible":
                    row[
                        "offer_eligible"
                    ],
                "product_url_raw":
                    row[
                        "product_url_raw"
                    ],
            }
        )


canonical_membership = pd.DataFrame(
    membership_rows
)

print(
    "Canonical membership rows:",
    f"{len(canonical_membership):,}"
)

display(
    canonical_membership.head(20)
)


Canonical membership rows: 1,720


,canonical_product_id,source_record_id,retailer,target_category,brand_normalized,product_name_raw,specification_raw,strict_pack_key,product_price,stock_status,currently_available,offer_eligible,product_url_raw
0,CP_0bfb1d33fb04,coldstorage_00004641,coldstorage,Snacks,fortune,Fortune Almond Soya Beancurd 360g,360g,g|1|360.000000|360.000000,1.85,IN_STOCK,True,True,https://coldstorage.com.sg/product/fortune-soy...
1,CP_0bfb1d33fb04,redmart_00000010,redmart,Snacks,fortune,Fortune Soya Beancurd Almond 360G,360 g,g|1|360.000000|360.000000,1.86,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i2366398545...
2,CP_f9d210d9f39d,coldstorage_00004642,coldstorage,Snacks,fortune,Fortune Original Soya Beancurd 360g,360g,g|1|360.000000|360.000000,1.85,IN_STOCK,True,True,https://coldstorage.com.sg/product/fortune-soy...
3,CP_f9d210d9f39d,redmart_00000018,redmart,Snacks,fortune,Fortune Soya Beancurd Original 360G,360 g,g|1|360.000000|360.000000,1.86,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i2366397508...
4,CP_44fe02df03cc,fairprice_00013004,fairprice,Snacks,glico,Glico Choco Mint Pocky,65 G,g|1|65.000000|65.000000,7.22,IN_STOCK,True,True,https://www.fairprice.com.sg/product/glico-cho...
5,CP_44fe02df03cc,redmart_00000024,redmart,Snacks,glico,Glico Pocky Choco Mint,65 g,g|1|65.000000|65.000000,5.20,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i380840606-...
6,CP_7a2ec66e892d,fairprice_00037252,fairprice,Snacks,vatan se,Vatan Se Malai Rabri,200 G,g|1|200.000000|200.000000,5.50,IN_STOCK,True,True,https://www.fairprice.com.sg/product/vatan-se-...
7,CP_7a2ec66e892d,redmart_00000037,redmart,Snacks,vatan se,Vatan Se Malai Rabri,200 g,g|1|200.000000|200.000000,5.55,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i1554424822...
8,CP_495c4f18420e,fairprice_00038596,fairprice,Snacks,fairy port,Fairy Port Frozen Musang King Durian Mille Cre...,400 G,g|1|400.000000|400.000000,28.99,IN_STOCK,True,True,https://www.fairprice.com.sg/product/fairy-por...
9,CP_495c4f18420e,redmart_00000894,redmart,Snacks,fairy port,Fairy Port Frozen Musang King Durian Mille Cre...,400 g,g|1|400.000000|400.000000,20.29,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i3203990851...


## 22. Attach fulfilment status to canonical membership

In [22]:
if len(canonical_membership) > 0:
    fulfilment_lookup = (
        fulfilment_review[
            [
                "source_record_id",
                "fulfilment_status",
                "fulfilment_precheck",
                "fulfilment_evidence",
            ]
        ]
        .drop_duplicates(
            subset=[
                "source_record_id"
            ]
        )
    )

    canonical_membership = (
        canonical_membership
        .merge(
            fulfilment_lookup,
            on="source_record_id",
            how="left",
        )
    )

    canonical_membership[
        "verified_direct_listing"
    ] = (
        canonical_membership[
            "fulfilment_status"
        ]
        == "VERIFIED_DIRECT"
    )

    canonical_membership[
        "verified_active_offer"
    ] = (
        canonical_membership[
            "verified_direct_listing"
        ]
        &
        canonical_membership[
            "offer_eligible"
        ]
        .astype("string")
        .str.casefold()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )

else:
    canonical_membership[
        "fulfilment_status"
    ] = pd.Series(
        dtype="string"
    )

    canonical_membership[
        "fulfilment_precheck"
    ] = pd.Series(
        dtype="string"
    )

    canonical_membership[
        "fulfilment_evidence"
    ] = pd.Series(
        dtype="string"
    )

    canonical_membership[
        "verified_direct_listing"
    ] = pd.Series(
        dtype="boolean"
    )

    canonical_membership[
        "verified_active_offer"
    ] = pd.Series(
        dtype="boolean"
    )


## 23. Check canonical group consistency and coverage

A connected component is identity-consistent only if all members share:

- one target category;
- one normalized brand;
- one strict pack key.

Multiple listings from the same retailer are retained but flagged for review because they may represent duplicates.


In [23]:
canonical_group_rows = []

if len(canonical_membership) > 0:
    for (
        canonical_product_id,
        group
    ) in canonical_membership.groupby(
        "canonical_product_id"
    ):
        category_count = (
            group[
                "target_category"
            ]
            .nunique(
                dropna=False
            )
        )

        brand_count = (
            group[
                "brand_normalized"
            ]
            .nunique(
                dropna=False
            )
        )

        pack_key_count = (
            group[
                "strict_pack_key"
            ]
            .nunique(
                dropna=False
            )
        )

        retailer_count = (
            group[
                "retailer"
            ]
            .nunique()
        )

        duplicate_retailer_count = int(
            (
                group[
                    "retailer"
                ]
                .value_counts()
                > 1
            ).sum()
        )

        verified_direct_retailers = (
            group.loc[
                group[
                    "verified_direct_listing"
                ]
                .fillna(False),
                "retailer"
            ]
            .nunique()
        )

        verified_active_offer_retailers = (
            group.loc[
                group[
                    "verified_active_offer"
                ]
                .fillna(False),
                "retailer"
            ]
            .nunique()
        )

        if (
            category_count != 1
            or brand_count != 1
            or pack_key_count != 1
        ):
            group_status = (
                "IDENTITY_CONFLICT"
            )

        elif (
            duplicate_retailer_count
            > 0
        ):
            group_status = (
                "REVIEW_DUPLICATE_RETAILER"
            )

        else:
            group_status = (
                "IDENTITY_VERIFIED"
            )

        canonical_group_rows.append(
            {
                "canonical_product_id":
                    canonical_product_id,
                "member_count":
                    len(group),
                "retailer_count":
                    retailer_count,
                "category_count":
                    category_count,
                "brand_count":
                    brand_count,
                "pack_key_count":
                    pack_key_count,
                "duplicate_retailer_count":
                    duplicate_retailer_count,
                "verified_direct_retailer_count":
                    verified_direct_retailers,
                "verified_active_offer_retailer_count":
                    verified_active_offer_retailers,
                "group_status":
                    group_status,
            }
        )


canonical_group_checks = pd.DataFrame(
    canonical_group_rows
)

print(
    "Canonical group status:"
)

if len(canonical_group_checks) > 0:
    print(
        canonical_group_checks[
            "group_status"
        ]
        .value_counts()
    )

    print(
        "\nVerified direct retailer coverage:"
    )

    print(
        canonical_group_checks[
            "verified_direct_retailer_count"
        ]
        .value_counts()
        .sort_index()
    )

else:
    print(
        "No canonical groups yet."
    )


Canonical group status:
group_status
IDENTITY_VERIFIED            816
REVIEW_DUPLICATE_RETAILER     17
Name: count, dtype: int64

Verified direct retailer coverage:
verified_direct_retailer_count
0    833
Name: count, dtype: int64


## 24. Create provisional canonical product table

In [24]:
canonical_product_rows = []

if len(canonical_group_checks) > 0:
    for check in (
        canonical_group_checks
        .itertuples(
            index=False
        )
    ):
        group = (
            canonical_membership.loc[
                canonical_membership[
                    "canonical_product_id"
                ]
                == check.canonical_product_id
            ]
        )

        if len(group) == 0:
            continue

        first = group.iloc[0]

        canonical_product_rows.append(
            {
                "canonical_product_id":
                    check.canonical_product_id,

                # Do not invent a display name.
                "canonical_name":
                    pd.NA,

                "target_category":
                    first[
                        "target_category"
                    ],
                "brand_normalized":
                    first[
                        "brand_normalized"
                    ],
                "strict_pack_key":
                    first[
                        "strict_pack_key"
                    ],
                "member_count":
                    check.member_count,
                "retailer_count":
                    check.retailer_count,
                "verified_direct_retailer_count":
                    check.verified_direct_retailer_count,
                "verified_active_offer_retailer_count":
                    check.verified_active_offer_retailer_count,
                "group_status":
                    check.group_status,

                "meets_two_retailer_identity_target":
                    (
                        check.retailer_count
                        >= 2
                    ),

                "meets_two_verified_direct_retailer_target":
                    (
                        check.verified_direct_retailer_count
                        >= 2
                    ),

                "canonical_review_notes":
                    pd.NA,
            }
        )


canonical_products = pd.DataFrame(
    canonical_product_rows
)

print(
    "Provisional canonical products:",
    f"{len(canonical_products):,}"
)

display(
    canonical_products.head(20)
)


Provisional canonical products: 833


,canonical_product_id,canonical_name,target_category,brand_normalized,strict_pack_key,member_count,retailer_count,verified_direct_retailer_count,verified_active_offer_retailer_count,group_status,meets_two_retailer_identity_target,meets_two_verified_direct_retailer_target,canonical_review_notes
0,CP_004eda4240c3,<NA>,Drinks,ogawa,ml|1|1000.000000|1000.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
1,CP_0062f6c9318a,<NA>,Drinks,kirin,ml|1|430.000000|430.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
2,CP_00c952729557,<NA>,Canned & Packaged Food,campbell's,ml|1|1000.000000|1000.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
3,CP_01769391e775,<NA>,Snacks,anand,g|1|150.000000|150.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
4,CP_01ae2e9feca4,<NA>,Snacks,sun q,g|1|170.000000|170.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
5,CP_01cbdf04620d,<NA>,Snacks,walkers,g|1|150.000000|150.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
6,CP_01e50d96fddb,<NA>,Snacks,snackfirst,g|1|150.000000|150.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
7,CP_01e79dc30fe3,<NA>,Snacks,calbee,g|1|180.000000|180.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
8,CP_01ef5221ac5c,<NA>,Snacks,calbee,g|1|72.000000|72.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>
9,CP_020a9d460350,<NA>,Canned & Packaged Food,green earth,g|1|110.000000|110.000000,2,2,0,0,IDENTITY_VERIFIED,True,False,<NA>


## 25. Preserve pack-discrepancy cases as a separate working file

The strict automated identity logic must not silently consume these cases.


In [25]:
PACK_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2c_pack_discrepancy_working.csv"
)

if len(pack_discrepancy) > 0:
    pack_review_base = (
        pack_discrepancy.copy()
    )

    pack_manual_columns = [
        "pack_review_decision",
        "corrected_source_record_id",
        "corrected_pack_count",
        "corrected_unit_size",
        "corrected_total_quantity_base",
        "correction_evidence",
        "correction_reviewer",
        "correction_notes",
    ]

    pack_review_base[
        "pack_review_decision"
    ] = "UNREVIEWED"

    for column in (
        pack_manual_columns[1:]
    ):
        pack_review_base[column] = pd.NA

    if PACK_REVIEW_FILE.exists():
        existing_pack = pd.read_csv(
            PACK_REVIEW_FILE,
            dtype="string",
            low_memory=False,
        )

        key_columns = [
            "left_source_record_id",
            "right_source_record_id",
        ]

        existing_pack_values = (
            existing_pack[
                [
                    *key_columns,
                    *pack_manual_columns,
                ]
            ]
            .drop_duplicates(
                subset=key_columns
            )
        )

        pack_review = (
            pack_review_base
            .drop(
                columns=pack_manual_columns
            )
            .merge(
                existing_pack_values,
                on=key_columns,
                how="left",
            )
        )

        pack_review[
            "pack_review_decision"
        ] = (
            pack_review[
                "pack_review_decision"
            ]
            .fillna("UNREVIEWED")
        )

    else:
        pack_review = (
            pack_review_base.copy()
        )

else:
    pack_review = (
        pack_discrepancy.copy()
    )

pack_review.to_csv(
    PACK_REVIEW_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Pack-discrepancy review rows:",
    f"{len(pack_review):,}"
)


Pack-discrepancy review rows: 281


## 26. Build automation and QA summary

In [26]:
automation_summary_rows = []

for decision, count in (
    automated_pairs[
        "auto_identity_decision"
    ]
    .value_counts()
    .items()
):
    automation_summary_rows.append(
        {
            "metric_type":
                "AUTO_IDENTITY_DECISION",
            "metric_name":
                decision,
            "count":
                int(count),
            "value":
                pd.NA,
        }
    )

for decision, count in (
    final_pairs[
        "final_identity_decision"
    ]
    .value_counts(
        dropna=False
    )
    .items()
):
    automation_summary_rows.append(
        {
            "metric_type":
                "FINAL_IDENTITY_DECISION",
            "metric_name":
                str(decision),
            "count":
                int(count),
            "value":
                pd.NA,
        }
    )

qa_reviewed_decision_count = int(
    final_pairs[
        "qa_has_decision"
    ].sum()
)

qa_complete_count = int(
    final_pairs[
        "qa_review_complete"
    ].sum()
)

qa_incomplete_count = int(
    final_pairs[
        "qa_review_incomplete"
    ].sum()
)

qa_exact_count = int(
    final_pairs[
        "qa_exact_label_agreement"
    ].sum()
)

qa_same_family_difference_count = int(
    final_pairs[
        "qa_same_family_label_difference"
    ].sum()
)

qa_family_agreement_count = int(
    final_pairs[
        "qa_family_agreement"
    ].sum()
)

qa_material_conflict_count = int(
    final_pairs[
        "qa_material_conflict"
    ].sum()
)

if qa_complete_count > 0:
    qa_family_agreement_rate = (
        qa_family_agreement_count
        / qa_complete_count
    )
else:
    qa_family_agreement_rate = np.nan

unresolved_material_conflict_count = len(
    unresolved_material_conflicts
)

automation_summary_rows.extend(
    [
        {
            "metric_type":
                "WORKLOAD",
            "metric_name":
                "TOTAL_CANDIDATE_PAIRS",
            "count":
                len(
                    automated_pairs
                ),
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "WORKLOAD",
            "metric_name":
                "MANUAL_REVIEW_REQUIRED",
            "count":
                int(
                    automated_pairs[
                        "requires_manual_review"
                    ].sum()
                ),
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "INDEPENDENT_QA_SAMPLE",
            "count":
                len(
                    qa_review
                ),
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_ROWS_WITH_DECISION",
            "count":
                qa_reviewed_decision_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_COMPLETE_REVIEWS",
            "count":
                qa_complete_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_INCOMPLETE_AUDIT_ROWS",
            "count":
                qa_incomplete_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_EXACT_LABEL_AGREEMENTS",
            "count":
                qa_exact_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_SAME_FAMILY_LABEL_DIFFERENCES",
            "count":
                qa_same_family_difference_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_FAMILY_AGREEMENTS",
            "count":
                qa_family_agreement_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_MATERIAL_CONFLICTS",
            "count":
                qa_material_conflict_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_UNRESOLVED_MATERIAL_CONFLICTS",
            "count":
                unresolved_material_conflict_count,
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "QA",
            "metric_name":
                "QA_FAMILY_AGREEMENT_RATE",
            "count":
                pd.NA,
            "value":
                (
                    round(
                        qa_family_agreement_rate,
                        6
                    )
                    if not pd.isna(
                        qa_family_agreement_rate
                    )
                    else pd.NA
                ),
        },
        {
            "metric_type":
                "RESULT",
            "metric_name":
                "IDENTITY_CONFIRMED_PAIRS",
            "count":
                len(
                    identity_confirmed_pairs
                ),
            "value":
                pd.NA,
        },
        {
            "metric_type":
                "RESULT",
            "metric_name":
                "PROVISIONAL_CANONICAL_PRODUCTS",
            "count":
                len(
                    canonical_products
                ),
            "value":
                pd.NA,
        },
    ]
)

automation_summary = pd.DataFrame(
    automation_summary_rows
)

display(
    automation_summary
)


,metric_type,metric_name,count,value
0,AUTO_IDENTITY_DECISION,NEEDS_MANUAL_REVIEW,7026,<NA>
1,AUTO_IDENTITY_DECISION,AUTO_NOT_SAME_PRODUCT,1602,<NA>
2,AUTO_IDENTITY_DECISION,AUTO_VARIANT_MISMATCH,1112,<NA>
3,AUTO_IDENTITY_DECISION,AUTO_MATCH_CONFIRMED,910,<NA>
4,FINAL_IDENTITY_DECISION,UNREVIEWED,7026,<NA>
5,FINAL_IDENTITY_DECISION,NOT_SAME_PRODUCT,1572,<NA>
6,FINAL_IDENTITY_DECISION,VARIANT_MISMATCH,1140,<NA>
7,FINAL_IDENTITY_DECISION,MATCH_CONFIRMED,912,<NA>
8,WORKLOAD,TOTAL_CANDIDATE_PAIRS,10650,<NA>
9,WORKLOAD,MANUAL_REVIEW_REQUIRED,7026,<NA>


## 27. Save all Stage 2C outputs

In [27]:
AUTOMATED_ASSESSMENT_FILE = (
    CURATED_DATA_DIR
    / "stage2c_automated_identity_assessment.csv"
)

FINAL_PAIRS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_final_identity_pairs.csv"
)

IDENTITY_CONFIRMED_FILE = (
    CURATED_DATA_DIR
    / "stage2c_identity_confirmed_pairs.csv"
)

QA_CONFLICTS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_conflicts.csv"
)

CANONICAL_MEMBERSHIP_FILE = (
    CURATED_DATA_DIR
    / "stage2c_canonical_membership_provisional.csv"
)

CANONICAL_GROUP_CHECKS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_canonical_group_checks.csv"
)

CANONICAL_PRODUCTS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_canonical_products_provisional.csv"
)

VERIFIED_DIRECT_LISTINGS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_verified_direct_listings.csv"
)

VERIFIED_ACTIVE_OFFERS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_verified_active_offers.csv"
)

AUTOMATION_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2c_automation_summary.csv"
)

QA_FAMILY_REPORT_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_family_agreement_report.csv"
)

QA_SAME_FAMILY_DIFFERENCES_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_same_family_label_differences.csv"
)

QA_MATERIAL_CONFLICTS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_material_conflicts.csv"
)

QA_INCOMPLETE_REVIEWS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_incomplete_reviews.csv"
)

QA_UNRESOLVED_CONFLICTS_FILE = (
    CURATED_DATA_DIR
    / "stage2c_qa_unresolved_material_conflicts.csv"
)

automated_pairs.to_csv(
    AUTOMATED_ASSESSMENT_FILE,
    index=False,
    encoding="utf-8",
)

final_pairs.to_csv(
    FINAL_PAIRS_FILE,
    index=False,
    encoding="utf-8",
)

identity_confirmed_pairs.to_csv(
    IDENTITY_CONFIRMED_FILE,
    index=False,
    encoding="utf-8",
)

qa_conflicts.to_csv(
    QA_CONFLICTS_FILE,
    index=False,
    encoding="utf-8",
)

canonical_membership.to_csv(
    CANONICAL_MEMBERSHIP_FILE,
    index=False,
    encoding="utf-8",
)

canonical_group_checks.to_csv(
    CANONICAL_GROUP_CHECKS_FILE,
    index=False,
    encoding="utf-8",
)

canonical_products.to_csv(
    CANONICAL_PRODUCTS_FILE,
    index=False,
    encoding="utf-8",
)

verified_direct_listings = (
    canonical_membership.loc[
        canonical_membership[
            "verified_direct_listing"
        ]
        .fillna(False)
    ]
    .copy()
)

verified_direct_listings.to_csv(
    VERIFIED_DIRECT_LISTINGS_FILE,
    index=False,
    encoding="utf-8",
)

verified_active_offers = (
    canonical_membership.loc[
        canonical_membership[
            "verified_active_offer"
        ]
        .fillna(False)
    ]
    .copy()
)

verified_active_offers.to_csv(
    VERIFIED_ACTIVE_OFFERS_FILE,
    index=False,
    encoding="utf-8",
)

automation_summary.to_csv(
    AUTOMATION_SUMMARY_FILE,
    index=False,
    encoding="utf-8",
)

qa_complete_rows.to_csv(
    QA_FAMILY_REPORT_FILE,
    index=False,
    encoding="utf-8",
)

qa_same_family_differences.to_csv(
    QA_SAME_FAMILY_DIFFERENCES_FILE,
    index=False,
    encoding="utf-8",
)

qa_material_conflicts.to_csv(
    QA_MATERIAL_CONFLICTS_FILE,
    index=False,
    encoding="utf-8",
)

qa_incomplete_reviews.to_csv(
    QA_INCOMPLETE_REVIEWS_FILE,
    index=False,
    encoding="utf-8",
)

unresolved_material_conflicts.to_csv(
    QA_UNRESOLVED_CONFLICTS_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Saved Stage 2C outputs."
)


Saved Stage 2C outputs.


## 28. Verify output files

In [28]:
OUTPUT_FILES = [
    AUTOMATED_ASSESSMENT_FILE,
    MANUAL_REVIEW_FILE,
    QA_REVIEW_FILE,
    FINAL_PAIRS_FILE,
    IDENTITY_CONFIRMED_FILE,
    QA_CONFLICTS_FILE,
    QA_CONFLICT_RESOLUTION_FILE,
    QA_FAMILY_REPORT_FILE,
    QA_SAME_FAMILY_DIFFERENCES_FILE,
    QA_MATERIAL_CONFLICTS_FILE,
    QA_INCOMPLETE_REVIEWS_FILE,
    QA_UNRESOLVED_CONFLICTS_FILE,
    FULFILMENT_REVIEW_FILE,
    PACK_REVIEW_FILE,
    CANONICAL_MEMBERSHIP_FILE,
    CANONICAL_GROUP_CHECKS_FILE,
    CANONICAL_PRODUCTS_FILE,
    VERIFIED_DIRECT_LISTINGS_FILE,
    VERIFIED_ACTIVE_OFFERS_FILE,
    AUTOMATION_SUMMARY_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:58s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print(
    "\nPASS: All automated Stage 2C output files exist."
)


stage2c_automated_identity_assessment.csv                  -> True
stage2c_manual_identity_review_working.csv                 -> True
stage2c_independent_qa_review_working.csv                  -> True
stage2c_final_identity_pairs.csv                           -> True
stage2c_identity_confirmed_pairs.csv                       -> True
stage2c_qa_conflicts.csv                                   -> True
stage2c_qa_conflict_resolution_working.csv                 -> True
stage2c_qa_family_agreement_report.csv                     -> True
stage2c_qa_same_family_label_differences.csv               -> True
stage2c_qa_material_conflicts.csv                          -> True
stage2c_qa_incomplete_reviews.csv                          -> True
stage2c_qa_unresolved_material_conflicts.csv               -> True
stage2c_fulfilment_review_working.csv                      -> True
stage2c_pack_discrepancy_working.csv                       -> True
stage2c_canonical_membership_provisional.csv               -> 

## 29. Final Stage 2C validation

In [29]:
total_pairs = len(
    automated_pairs
)

auto_confirmed_count = int(
    (
        automated_pairs[
            "auto_identity_decision"
        ]
        == "AUTO_MATCH_CONFIRMED"
    ).sum()
)

auto_variant_mismatch_count = int(
    (
        automated_pairs[
            "auto_identity_decision"
        ]
        == "AUTO_VARIANT_MISMATCH"
    ).sum()
)

auto_not_same_count = int(
    (
        automated_pairs[
            "auto_identity_decision"
        ]
        == "AUTO_NOT_SAME_PRODUCT"
    ).sum()
)

manual_required_count = int(
    automated_pairs[
        "requires_manual_review"
    ].sum()
)

qa_reviewed_decision_count = int(
    final_pairs[
        "qa_has_decision"
    ].sum()
)

qa_complete_count = int(
    final_pairs[
        "qa_review_complete"
    ].sum()
)

qa_incomplete_count = int(
    final_pairs[
        "qa_review_incomplete"
    ].sum()
)

qa_exact_count = int(
    final_pairs[
        "qa_exact_label_agreement"
    ].sum()
)

qa_same_family_difference_count = int(
    final_pairs[
        "qa_same_family_label_difference"
    ].sum()
)

qa_family_agreement_count = int(
    final_pairs[
        "qa_family_agreement"
    ].sum()
)

qa_material_conflict_count = int(
    final_pairs[
        "qa_material_conflict"
    ].sum()
)

if qa_complete_count > 0:
    qa_family_agreement_rate = (
        qa_family_agreement_count
        / qa_complete_count
    )
else:
    qa_family_agreement_rate = np.nan

unresolved_material_conflict_count = len(
    unresolved_material_conflicts
)

final_confirmed_count = len(
    identity_confirmed_pairs
)

canonical_product_count = len(
    canonical_products
)

verified_direct_listing_count = len(
    verified_direct_listings
)

verified_active_offer_count = len(
    verified_active_offers
)

print("=" * 98)
print("STAGE 2C FINAL VALIDATION — REVISED QA FAMILY LOGIC")
print("=" * 98)

print(
    f"Total Stage 2B candidate pairs                : "
    f"{total_pairs:,}"
)

print("-" * 98)

print(
    f"AUTO_MATCH_CONFIRMED                         : "
    f"{auto_confirmed_count:,}"
)

print(
    f"AUTO_VARIANT_MISMATCH                        : "
    f"{auto_variant_mismatch_count:,}"
)

print(
    f"AUTO_NOT_SAME_PRODUCT                        : "
    f"{auto_not_same_count:,}"
)

print(
    f"NEEDS_MANUAL_REVIEW                          : "
    f"{manual_required_count:,}"
)

print("-" * 98)

print(
    f"Independent QA sample                        : "
    f"{len(qa_review):,}"
)

print(
    f"QA rows with decision                        : "
    f"{qa_reviewed_decision_count:,}"
)

print(
    f"QA complete reviews                          : "
    f"{qa_complete_count:,}"
)

print(
    f"QA incomplete audit rows                     : "
    f"{qa_incomplete_count:,}"
)

print(
    f"QA exact-label agreements                    : "
    f"{qa_exact_count:,}"
)

print(
    f"QA same-family label differences             : "
    f"{qa_same_family_difference_count:,}"
)

print(
    f"QA family-level agreements                   : "
    f"{qa_family_agreement_count:,}"
)

print(
    f"QA material family conflicts                 : "
    f"{qa_material_conflict_count:,}"
)

if qa_complete_count > 0:
    print(
        f"QA family-level agreement rate               : "
        f"{qa_family_agreement_rate:.2%}"
    )

print(
    f"Unresolved material QA conflicts             : "
    f"{unresolved_material_conflict_count:,}"
)

print("-" * 98)

print(
    f"Final MATCH_CONFIRMED pairs                  : "
    f"{final_confirmed_count:,}"
)

print(
    f"Provisional canonical products               : "
    f"{canonical_product_count:,}"
)

print(
    f"Verified direct canonical listings           : "
    f"{verified_direct_listing_count:,}"
)

print(
    f"Verified active offers                       : "
    f"{verified_active_offer_count:,}"
)

print(
    f"Fulfilment listings awaiting/checking        : "
    f"{len(fulfilment_review):,}"
)

print("-" * 98)

print(
    f"Pack-discrepancy review rows                 : "
    f"{len(pack_review):,}"
)

print("=" * 98)

assert (
    auto_confirmed_count
    + auto_variant_mismatch_count
    + auto_not_same_count
    + manual_required_count
    == total_pairs
)

assert (
    automated_pairs[
        "candidate_pair_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    qa_exact_count
    + qa_same_family_difference_count
    + qa_material_conflict_count
    == qa_complete_count
), (
    "Every complete QA review must be either "
    "exact agreement, same-family label difference, "
    "or a material family conflict."
)

assert (
    qa_family_agreement_count
    == (
        qa_exact_count
        + qa_same_family_difference_count
    )
)

assert (
    final_confirmed_count
    <= total_pairs
)

if len(canonical_membership) > 0:
    assert (
        canonical_membership[
            "canonical_product_id"
        ]
        .notna()
        .all()
    )

print()
print(
    "PASS: Revised automated Stage 2C identity assessment, "
    "QA family validation, conflict handling, and provisional "
    "canonical grouping succeeded."
)

print()

if qa_incomplete_count > 0:
    print(
        "ACTION: Complete qa_reviewer and qa_evidence for "
        "the QA rows that already have a decision."
    )

if unresolved_material_conflict_count > 0:
    print(
        "ACTION: Resolve only the material family conflicts in "
        "stage2c_qa_conflict_resolution_working.csv."
    )

if manual_required_count > 0:
    print(
        "OPTIONAL: Review stage2c_manual_identity_review_working.csv "
        "only if more canonical candidates are needed."
    )

print(
    "For selected/final MATCH_CONFIRMED listings, verify direct "
    "supermarket fulfilment in "
    "stage2c_fulfilment_review_working.csv."
)


STAGE 2C FINAL VALIDATION — REVISED QA FAMILY LOGIC
Total Stage 2B candidate pairs                : 10,650
--------------------------------------------------------------------------------------------------
AUTO_MATCH_CONFIRMED                         : 910
AUTO_VARIANT_MISMATCH                        : 1,112
AUTO_NOT_SAME_PRODUCT                        : 1,602
NEEDS_MANUAL_REVIEW                          : 7,026
--------------------------------------------------------------------------------------------------
Independent QA sample                        : 90
QA rows with decision                        : 90
QA complete reviews                          : 90
QA incomplete audit rows                     : 0
QA exact-label agreements                    : 60
QA same-family label differences             : 28
QA family-level agreements                   : 88
QA material family conflicts                 : 2
QA family-level agreement rate               : 97.78%
Unresolved material QA conflicts 

## How to use the revised automated Stage 2C workflow

### 1. Run the notebook

The automated identity rules classify all strict Stage 2B candidate pairs into:

```text
AUTO_MATCH_CONFIRMED
AUTO_VARIANT_MISMATCH
AUTO_NOT_SAME_PRODUCT
NEEDS_MANUAL_REVIEW
```

### 2. Complete the independent QA sample

Open:

```text
stage2c_independent_qa_review_working.csv
```

For every independently reviewed row, fill:

```text
qa_identity_decision
qa_reviewer
qa_evidence
qa_notes
```

`qa_reviewer` should identify the actual reviewer, for example:

```text
OSH
Member 2
Alice Tan
```

`qa_evidence` should briefly state the evidence used, for example:

```text
Same brand, same Zero Sugar variant and exact 6 x 320 ml pack; retailer wording only differs.
```

A QA decision is considered complete only when the decision, reviewer and evidence are present.

### 3. Understand the revised QA metric

The notebook evaluates decisions by family:

```text
MATCH:
    MATCH_CONFIRMED

NON_MATCH:
    VARIANT_MISMATCH
    NOT_SAME_PRODUCT
    EXCLUDE_OTHER

UNRESOLVED:
    NEEDS_MORE_EVIDENCE
    UNREVIEWED
```

Therefore:

```text
AUTO_NOT_SAME_PRODUCT
vs
QA VARIANT_MISMATCH
```

is counted as **QA agreement at the family level**, not as a material conflict.

The notebook reports:

```text
QA exact-label agreements
QA same-family label differences
QA family-level agreements
QA material family conflicts
QA family-level agreement rate
```

### 4. Resolve only material QA conflicts

If there are material family conflicts, open:

```text
stage2c_qa_conflict_resolution_working.csv
```

Fill:

```text
resolution_identity_decision
resolution_reviewer
resolution_evidence
resolution_notes
```

Only conflicts such as `MATCH` versus `NON_MATCH` enter this file.

After completing the conflict-resolution rows, rerun the notebook.

### 5. Manual-review queue is optional unless more candidates are required

The file:

```text
stage2c_manual_identity_review_working.csv
```

contains the borderline automated cases.

Do **not** automatically review all of them. Review them only if the high-confidence confirmed matches do not provide enough canonical products or sufficient category/retailer coverage.

### 6. Fulfilment verification remains separate

For source listings involved in confirmed identity pairs, use:

```text
stage2c_fulfilment_review_working.csv
```

Allowed statuses are:

```text
VERIFIED_DIRECT
NOT_DIRECT
UNKNOWN
UNREVIEWED
```

The URL/domain precheck helps prioritize verification but does not automatically prove direct supermarket fulfilment.

### 7. Stock remains separate from identity

An out-of-stock listing can still belong to the correct canonical product.

It becomes a verified active offer only when:

```text
fulfilment_status = VERIFIED_DIRECT
```

and its Stage 1E `offer_eligible` value is also true.

### 8. Pack-discrepancy cases remain separate

Do not merge:

```text
stage2c_pack_discrepancy_working.csv
```

cases merely because total quantity is equal. They require explicit evidence or a source-specification correction.

### Next stage

Once:

- the independent QA sample is complete;
- material QA conflicts are resolved;
- the automated pilot has an acceptable family-level agreement rate;

proceed to **Stage 2D: expand the validated identity-matching process from the three pilot categories to all eight CartOptix categories**.

Fulfilment verification can then be focused on the final balanced 200–300 canonical-product seed rather than every provisional pilot listing.
